# Lecture: Evaluating Generative Models I — Quantitative Metrics

Throughout this course we judged generated images **by eye**: "the GAN looks
sharper than the VAE", "the DDPM samples are diverse". That is subjective and does
not scale. This notebook introduces **quantitative metrics** that put a *number* on
generation quality, and uses them to compare the three Fashion-MNIST models you
trained:

- the **GAN** (C3-1),
- the **DDPM** (pixel-space diffusion, C4-2),
- the **Latent Diffusion** model (C4-5).

The central metric is the **Fréchet Inception Distance (FID)** — the standard for
image generation. We will also look at what FID *cannot* tell us, motivating the
**precision/recall** view of quality vs. diversity.

> **A caveat up front.** FID was designed for natural RGB images and uses an
> InceptionV3 network trained on ImageNet. Our images are 28×28 grayscale clothing
> items, far outside Inception's training domain, so the **absolute** FID values
> here are not meaningful in the usual sense. What *is* meaningful is the
> **relative ranking** of our three models under identical conditions — which is
> exactly the didactic point.

This notebook runs fine on **CPU** (the models and images are tiny); a GPU just
makes it faster.

Run the following cell only if you are working with Google Colab to copy the required .py files into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C3-GANs/GAN.py ./
!cp AIBIP/C4-Diffusion_Models/Diffusion.py ./
!cp AIBIP/C4-Diffusion_Models/AutoEncoder.py ./
!pip install -q "torchmetrics[image]"

### Why we cannot just compare pixels

A naive idea would be to measure how close generated images are to real ones
pixel-by-pixel (e.g. MSE). This fails completely for generative models: a perfect
sample that happens to be a *different* (but valid) shirt than any specific real
image would score a huge pixel error. We do not want per-image matching — we want
to compare **distributions**.

FID does exactly that. It passes both real and generated images through a fixed
feature extractor (InceptionV3), models each set of features as a multivariate
Gaussian, and measures the **Fréchet distance** between the two Gaussians:

$$\text{FID} = \lVert \mu_r - \mu_g \rVert^2 + \mathrm{Tr}\!\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

Lower is better: it means the generated feature distribution matches the real one
in both **mean** (typical content) and **covariance** (variation).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from torchmetrics.image.fid import FrechetInceptionDistance

from GAN import GAN
from Diffusion import DDPM, LatentDDPM
from AutoEncoder import AutoEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Fashion-MNIST in [-1, 1], the convention used by all our models.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
dataset = FashionMNIST(root="./data", train=False, download=True, transform=transform)
print("Test images:", len(dataset))

### Preparing images for FID

`torchmetrics`' FID expects **uint8 RGB** images. Our samples are grayscale in
$[-1, 1]$, so we convert them: rescale to $[0, 255]$, cast to `uint8`, and repeat
the single channel three times to make RGB. (InceptionV3 resizes to 299×299
internally, so we do not need to.)

In [ ]:
def to_fid_format(imgs):
    """Convert (N,1,28,28) images in [-1,1] to (N,3,28,28) uint8 RGB for FID."""
    imgs = (imgs.clamp(-1, 1) + 1) / 2          # [-1,1] -> [0,1]
    imgs = (imgs * 255).to(torch.uint8)          # -> [0,255] uint8
    return imgs.repeat(1, 3, 1, 1)               # grayscale -> RGB

### Load the three models

We load each pre-trained model with the same configuration used in its notebook.
The Latent Diffusion model additionally needs its autoencoder to decode latents
back to images.

In [ ]:
# GAN (C3-1)
gan = GAN(latent_dim=64, channels=64).to(device)
gan.load_model(path="AIBIP/C3-GANs/models/gan_fashion_mnist.pth", device=device)
gan.eval()

# DDPM (C4-2), 50-epoch checkpoint
ddpm = DDPM(timesteps=1000, channels=64).to(device)
ddpm.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_50epochs.pth", device=device)
ddpm.eval()

# Latent Diffusion (C4-5): autoencoder + conditional latent DDPM
ae = AutoEncoder(latent_channels=4, channels=32).to(device)
ae.load_model(path="AIBIP/C4-Diffusion_Models/models/autoencoder_fashion_mnist.pth", device=device)
ae.eval()

ldm = LatentDDPM(latent_channels=4, timesteps=1000, channels=128, num_classes=10).to(device)
ldm.load_model(path="AIBIP/C4-Diffusion_Models/models/latent_ddpm_fashion_mnist.pth", device=device)
ldm.eval()

### Generate samples from each model

We draw the same number of samples from each model. To keep the notebook fast on
CPU we use a modest sample count and DDIM (50 steps) for the diffusion models —
feel free to raise `N_SAMPLES` for a more stable FID if you have a GPU.

The Latent Diffusion model is class-conditional, so we generate an equal number of
each class for a fair, balanced sample (and decode the latents with the
autoencoder).

In [ ]:
N_SAMPLES = 1000   # raise to 5000-10000 on a GPU for a more stable FID

@torch.no_grad()
def gen_gan(n):
    return gan.generate(n, device=device)

@torch.no_grad()
def gen_ddpm(n):
    return ddpm.ddim_sample(n, steps=50, device=device)

@torch.no_grad()
def gen_ldm(n):
    # balanced labels across the 10 classes
    y = torch.arange(n, device=device) % 10
    z = ldm.sample_cfg(y, steps=50, guidance_scale=3.0, device=device)
    return ae.decode(z)

print("Generating samples (this is the slow part on CPU)...")
samples = {
    "GAN":             gen_gan(N_SAMPLES),
    "DDPM":            gen_ddpm(N_SAMPLES),
    "LatentDiffusion": gen_ldm(N_SAMPLES),
}
for name, s in samples.items():
    print(f"  {name}: {tuple(s.shape)}")

### A quick visual sanity check

Before trusting the numbers, glance at a few samples from each model. The FID
ranking should roughly agree with what you see here.

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))
for row, (name, s) in enumerate(samples.items()):
    imgs = (s[:8].clamp(-1, 1).cpu() + 1) / 2
    for col in range(8):
        axes[row, col].imshow(imgs[col].squeeze(), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(name, rotation=0, labelpad=45, fontsize=11, va="center")
plt.suptitle("Samples from each model", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### Compute FID for each model

For each model we feed the **real** test images and the **generated** images into a
fresh FID metric, then read off the distance. We use a batch of real images of the
same size as our generated set.

Lower FID = generated distribution closer to the real one.

In [ ]:
# Collect a batch of real images once.
real_loader = DataLoader(dataset, batch_size=256, shuffle=True)
real_imgs = []
collected = 0
for x, _ in real_loader:
    real_imgs.append(x)
    collected += x.size(0)
    if collected >= N_SAMPLES:
        break
real_imgs = torch.cat(real_imgs)[:N_SAMPLES]
real_fid = to_fid_format(real_imgs)

fid_scores = {}
for name, s in samples.items():
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    fid.update(real_fid.to(device), real=True)
    fid.update(to_fid_format(s.cpu()).to(device), real=False)
    fid_scores[name] = fid.compute().item()
    print(f"{name:16s} FID = {fid_scores[name]:.2f}")

### The verdict

Plotting the FID scores side by side gives the quantitative ranking of our three
generative models — the first time in this course we compare them with a number
rather than an opinion.

In [ ]:
names = list(fid_scores.keys())
values = [fid_scores[n] for n in names]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, values, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_ylabel("FID (lower is better)")
ax.set_title("Fréchet Inception Distance by model — Fashion-MNIST")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.1f}",
            ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

### What FID does *not* tell you: quality vs. diversity

A single FID number conflates two very different failure modes:

- **Low quality** — samples look bad/unrealistic.
- **Low diversity** — samples look good but are all similar (e.g. GAN **mode
  collapse**, C3-1). A model that produces 10 perfect but identical shirts can
  still score a deceptively reasonable FID.

The **precision/recall** framework separates these: *precision* measures how
realistic samples are (quality), *recall* measures how much of the real
distribution they cover (diversity). As a lightweight proxy here, we measure the
**per-pixel variance** across each model's samples — the same diversity probe used
in the GAN mode-collapse cell (C3-1). Higher variance ⇒ more diverse output.

In [ ]:
print(f"{'Model':16s} {'mean per-pixel variance':>24s}")
print("-" * 42)
for name, s in samples.items():
    var = s.var(dim=0).mean().item()
    print(f"{name:16s} {var:>24.4f}")

print("\n(Very low variance suggests mode collapse / limited diversity,")
print(" which FID alone may not fully penalise.)")

## Summary

| Metric | Measures | Limitation |
|---|---|---|
| Pixel MSE | per-image distance | meaningless for generation (no 1:1 match) |
| **FID** | distance between feature *distributions* | one number; domain-sensitive; hides diversity issues |
| Precision / Recall | quality vs. coverage separately | needs more samples; harder to compute |
| Per-pixel variance | crude diversity proxy | ignores quality entirely |

The lesson: **no single number captures "good generation"**. FID is the standard
and a good default, but it must be read alongside a diversity measure and, always,
a visual check. With these tools you can now compare generative models
*objectively* — the missing piece after building them in C1–C5.

The final notebook (C6-2) turns from *how good* models are to *what risks* they
bring: bias, deepfakes, and responsible use.

---
## Try It Yourself — Measuring Generation Quality

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Does the ranking match your eyes?** Compare the FID bar chart to the visual
sample grid. Does the model with the best (lowest) FID also look best to you? Where
do metric and intuition disagree, and why might that happen on grayscale data?

**B. Sample size matters.** Re-run with `N_SAMPLES = 200` and then a larger value.
How much does FID move? Why is FID **biased** at small sample sizes, and what does
that imply about comparing published FID numbers?

**C. FID vs. diversity.** Look at the per-pixel variance table next to the FID
scores. Is the lowest-FID model also the most diverse? Construct an argument for
why a model could "cheat" FID by sacrificing diversity.

**D. Break FID deliberately.** Feed the **real** images as *both* real and fake to
the metric. What FID do you get, and why? Now add a tiny bit of noise to the fake
set — does FID rise? What does this tell you about its sensitivity.

**E. Beyond one number.** Propose (in words) an evaluation protocol you would
actually trust to compare two image generators. Which combination of metrics and
checks would you require, and why is no single one sufficient?